# JAX/Flax/Optax Addition Transformer

A roughly 10M-parameter causal Transformer trained on fixed-length character addition examples. The vocabulary is hard-coded to digits `0-9`, space, `+`, and `=`. Select a Colab TPU or T4 GPU runtime before running.


In [ ]:
"""
Colab-ready JAX/Flax/Optax character Transformer for 3-digit addition.

Runtime target: free Colab TPU or T4 GPU. In Colab, choose
Runtime -> Change runtime type -> TPU or T4 GPU, then run this cell.
"""

import subprocess
import sys
import time
from functools import partial

try:
    import jax
    import jax.numpy as jnp
    import flax.linen as nn
    from flax.training import train_state
    import optax
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "flax", "optax"])
    import jax
    import jax.numpy as jnp
    import flax.linen as nn
    from flax.training import train_state
    import optax


jax.config.update("jax_default_matmul_precision", "tensorfloat32")

VOCAB = "0123456789 +="
assert set(VOCAB) == set("0123456789 +=")
CHAR_TO_ID = {ch: i for i, ch in enumerate(VOCAB)}
ID_TO_CHAR = {i: ch for ch, i in CHAR_TO_ID.items()}

SPACE = CHAR_TO_ID[" "]
PLUS = CHAR_TO_ID["+"]
EQ = CHAR_TO_ID["="]
VOCAB_SIZE = len(VOCAB)

WIDTH = 3
RESULT_WIDTH = 4
SEQ_LEN = WIDTH + 1 + WIDTH + 1 + RESULT_WIDTH
INPUT_LEN = SEQ_LEN - 1
ANSWER_TARGET_START = WIDTH + 1 + WIDTH

N_LAYERS = 6
D_MODEL = 384
N_HEADS = 6
D_FF = 1536

DEVICE_PLATFORM = jax.devices()[0].platform
BATCH_SIZE = 8192 if DEVICE_PLATFORM == "tpu" else 4096
EVAL_BATCH_SIZE = BATCH_SIZE
STEPS = 1500
WARMUP_STEPS = 100
LEARNING_RATE = 3e-4
LOG_EVERY = 100

print("JAX devices:", jax.devices())
print(f"Vocabulary: {repr(VOCAB)} ({VOCAB_SIZE} tokens)")
print(f"Fixed sequence length: {SEQ_LEN}, batch size: {BATCH_SIZE}")


def digits3(n):
    n = n.astype(jnp.int32)
    hundreds = (n // 100) % 10
    tens = (n // 10) % 10
    ones = n % 10
    return jnp.stack(
        [
            jnp.where(n >= 100, hundreds, SPACE),
            jnp.where(n >= 10, tens, SPACE),
            ones,
        ],
        axis=-1,
    ).astype(jnp.int32)


def digits4(n):
    n = n.astype(jnp.int32)
    thousands = (n // 1000) % 10
    hundreds = (n // 100) % 10
    tens = (n // 10) % 10
    ones = n % 10
    return jnp.stack(
        [
            jnp.where(n >= 1000, thousands, SPACE),
            jnp.where(n >= 100, hundreds, SPACE),
            jnp.where(n >= 10, tens, SPACE),
            ones,
        ],
        axis=-1,
    ).astype(jnp.int32)


@partial(jax.jit, static_argnames=("batch_size",))
def make_batch(key, batch_size):
    key_a, key_b = jax.random.split(key)
    a = jax.random.randint(key_a, (batch_size,), minval=0, maxval=1000, dtype=jnp.int32)
    b = jax.random.randint(key_b, (batch_size,), minval=0, maxval=1000, dtype=jnp.int32)
    seq = jnp.concatenate(
        [
            digits3(a),
            jnp.full((batch_size, 1), PLUS, dtype=jnp.int32),
            digits3(b),
            jnp.full((batch_size, 1), EQ, dtype=jnp.int32),
            digits4(a + b),
        ],
        axis=-1,
    )
    x = seq[:, :-1]
    y = seq[:, 1:]
    answer_mask = (jnp.arange(INPUT_LEN) >= ANSWER_TARGET_START)[None, :]
    return x, y, answer_mask, a, b, seq


class CausalSelfAttention(nn.Module):
    d_model: int
    n_heads: int

    @nn.compact
    def __call__(self, x):
        batch, seq_len, channels = x.shape
        head_dim = self.d_model // self.n_heads
        assert self.d_model % self.n_heads == 0

        q = nn.Dense(self.d_model, use_bias=False, name="q")(x)
        k = nn.Dense(self.d_model, use_bias=False, name="k")(x)
        v = nn.Dense(self.d_model, use_bias=False, name="v")(x)

        q = q.reshape(batch, seq_len, self.n_heads, head_dim)
        k = k.reshape(batch, seq_len, self.n_heads, head_dim)
        v = v.reshape(batch, seq_len, self.n_heads, head_dim)

        attn_logits = jnp.einsum("bthd,bshd->bhts", q, k) * (head_dim**-0.5)
        causal_mask = jnp.tril(jnp.ones((seq_len, seq_len), dtype=bool))
        attn_logits = jnp.where(causal_mask[None, None, :, :], attn_logits, -1.0e9)
        attn = nn.softmax(attn_logits, axis=-1)
        out = jnp.einsum("bhts,bshd->bthd", attn, v)
        out = out.reshape(batch, seq_len, channels)
        return nn.Dense(self.d_model, use_bias=False, name="out")(out)


class TransformerBlock(nn.Module):
    d_model: int
    n_heads: int
    d_ff: int

    @nn.compact
    def __call__(self, x):
        x = x + CausalSelfAttention(self.d_model, self.n_heads, name="attn")(
            nn.LayerNorm(name="ln_attn")(x)
        )
        h = nn.LayerNorm(name="ln_mlp")(x)
        h = nn.Dense(self.d_ff, name="fc1")(h)
        h = nn.gelu(h)
        h = nn.Dense(self.d_model, name="fc2")(h)
        return x + h


class TinyAdditionTransformer(nn.Module):
    vocab_size: int = VOCAB_SIZE
    max_len: int = INPUT_LEN
    d_model: int = D_MODEL
    n_heads: int = N_HEADS
    d_ff: int = D_FF
    n_layers: int = N_LAYERS

    @nn.compact
    def __call__(self, idx, train=False):
        seq_len = idx.shape[1]
        x = nn.Embed(self.vocab_size, self.d_model, name="token_embedding")(idx)
        pos = self.param(
            "position_embedding",
            nn.initializers.normal(stddev=0.02),
            (1, self.max_len, self.d_model),
        )
        x = x + pos[:, :seq_len, :]
        for layer in range(self.n_layers):
            x = TransformerBlock(self.d_model, self.n_heads, self.d_ff, name=f"block_{layer}")(x)
        x = nn.LayerNorm(name="final_ln")(x)
        return nn.Dense(self.vocab_size, name="lm_head")(x)


def count_params(params):
    return sum(x.size for x in jax.tree_util.tree_leaves(params))


def masked_answer_loss(logits, labels, mask):
    per_token = optax.softmax_cross_entropy_with_integer_labels(logits, labels)
    mask = jnp.broadcast_to(mask, labels.shape)
    return (per_token * mask).sum() / mask.sum()


def answer_metrics(logits, labels, mask):
    pred = jnp.argmax(logits, axis=-1)
    mask = jnp.broadcast_to(mask, labels.shape)
    token_acc = ((pred == labels) * mask).sum() / mask.sum()
    exact_acc = jnp.all(
        pred[:, ANSWER_TARGET_START : ANSWER_TARGET_START + RESULT_WIDTH]
        == labels[:, ANSWER_TARGET_START : ANSWER_TARGET_START + RESULT_WIDTH],
        axis=-1,
    ).mean()
    return token_acc, exact_acc


model = TinyAdditionTransformer()
rng = jax.random.PRNGKey(0)
rng, init_key, sample_key = jax.random.split(rng, 3)
sample_x, _, _, _, _, _ = make_batch(sample_key, 8)
params = model.init(init_key, sample_x, train=True)["params"]
param_count = count_params(params)
print(f"Parameter count: {param_count:,} ({param_count / 1e6:.2f}M)")

schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    decay_steps=STEPS,
    end_value=LEARNING_RATE * 0.1,
)
tx = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adamw(learning_rate=schedule, weight_decay=0.01),
)
state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)


@jax.jit
def train_step(state, key):
    x, y, mask, _, _, _ = make_batch(key, BATCH_SIZE)

    def loss_fn(params):
        logits = state.apply_fn({"params": params}, x, train=True)
        loss = masked_answer_loss(logits, y, mask)
        token_acc, exact_acc = answer_metrics(logits, y, mask)
        return loss, (token_acc, exact_acc)

    (loss, (token_acc, exact_acc)), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
    state = state.apply_gradients(grads=grads)
    return state, {"loss": loss, "token_acc": token_acc, "exact_acc": exact_acc}


@jax.jit
def eval_step(state, key):
    x, y, mask, _, _, _ = make_batch(key, EVAL_BATCH_SIZE)
    logits = state.apply_fn({"params": state.params}, x, train=False)
    loss = masked_answer_loss(logits, y, mask)
    token_acc, exact_acc = answer_metrics(logits, y, mask)
    return {"loss": loss, "token_acc": token_acc, "exact_acc": exact_acc}


start = time.time()
for step in range(1, STEPS + 1):
    rng, step_key = jax.random.split(rng)
    state, train_metrics = train_step(state, step_key)

    if step == 1 or step % LOG_EVERY == 0:
        rng, eval_key = jax.random.split(rng)
        eval_metrics = jax.device_get(eval_step(state, eval_key))
        elapsed = time.time() - start
        print(
            f"step {step:4d} | "
            f"eval loss {eval_metrics['loss']:.4f} | "
            f"token acc {eval_metrics['token_acc']:.4f} | "
            f"exact acc {eval_metrics['exact_acc']:.4f} | "
            f"{elapsed:.1f}s"
        )


def number_tokens3_py(n):
    return [
        SPACE if n < 100 else n // 100,
        SPACE if n < 10 else (n // 10) % 10,
        n % 10,
    ]


def ids_to_text(ids):
    return "".join(ID_TO_CHAR[int(i)] for i in ids)


def predict_addition(a, b):
    ids = number_tokens3_py(a) + [PLUS] + number_tokens3_py(b) + [EQ]
    for _ in range(RESULT_WIDTH):
        x = jnp.array([ids], dtype=jnp.int32)
        logits = model.apply({"params": state.params}, x, train=False)
        next_id = int(jnp.argmax(logits[0, -1]))
        ids.append(next_id)
    return ids_to_text(ids)


print("\nGreedy samples:")
for a, b in [(7, 42), (99, 1), (123, 456), (908, 77), (999, 999)]:
    print(f"{a:3d} + {b:3d} -> {predict_addition(a, b)!r}")
